# DMorphNet — Face Morphing Detection
### EfficientNet-B6 features + SVM classifier

A clean, end-to-end implementation of *DMorphNet* (Gawade et al.). A **morphed**
face blends two identities into one image that can fool face verification. This
notebook builds a dataset, extracts deep features with **EfficientNet-B6**, and
classifies **real vs. morph** with an **SVM**.

**Pipeline**

`face image → resize 528 + CLAHE → EfficientNet-B6 → 2304-d vector → SVM → Real / Morph`

Why two stages? A frozen CNN gives strong features while an SVM draws a robust
boundary — this generalizes better than one end-to-end network on a small dataset.

> Run the cells top to bottom. Enable a GPU first: **Runtime → Change runtime type → GPU**.


## 1 · Setup

Install the libraries and confirm a GPU is available.


In [ ]:
!pip -q install kagglehub mediapipe opencv-python-headless scikit-learn tqdm

import os, glob, random, urllib.request
import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__,
      "| GPU:", "yes" if tf.config.list_physical_devices('GPU') else "NO (enable it)")


## 2 · Configuration

`DEMO = True` runs a small, fast dataset that still produces real results
(~20 min on a GPU). Set `DEMO = False` for the paper's full scale.


In [ ]:
DEMO = True

if DEMO:
    N_REAL, N_MORPH = 900, 700          # quick but real
else:
    N_REAL, N_MORPH = 24000, 21000      # paper scale

IMG_SIZE   = 528        # EfficientNet-B6 input
MORPH_SIZE = 256        # morph canvas
ALPHA      = 0.5        # 50/50 identity blend

DATA_DIR  = "/content/dmorphnet"
MORPH_DIR = f"{DATA_DIR}/morph"
os.makedirs(MORPH_DIR, exist_ok=True)
print(f"Dataset target: {N_REAL} real + {N_MORPH} morph = {N_REAL + N_MORPH} images")


## 3 · Real faces (FFHQ)

The **real** class comes from **FFHQ** — thousands of distinct, high-quality
faces. We use a 256-px mirror so it downloads quickly on Colab.


In [ ]:
import kagglehub

FFHQ = kagglehub.dataset_download("xhlulu/flickrfaceshq-dataset-nvidia-resized-256px")
face_paths = [p for p in glob.glob(os.path.join(FFHQ, "**", "*"), recursive=True)
              if p.lower().endswith((".png", ".jpg", ".jpeg"))]
random.shuffle(face_paths)
print("FFHQ faces available:", len(face_paths))

# preview
fig, ax = plt.subplots(1, 5, figsize=(14, 3))
for a, p in zip(ax, face_paths[:5]):
    a.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)); a.axis("off")
fig.suptitle("Real faces (FFHQ)"); plt.tight_layout(); plt.show()


## 4 · Morph generation

Morphs are created by **landmark-based morphing** (the standard face-morphing
recipe): find 478 facial landmarks on both faces, warp each to the *average*
shape with a Delaunay triangle mesh, blend the two, and seamlessly clone the
blended face onto one photo so hair and background stay clean.

`average shape:  p̄ = ½·p₁ + ½·p₂`


In [ ]:
import mediapipe as mp
from mediapipe.tasks.python import vision, BaseOptions

_MODEL = "/content/face_landmarker.task"
if not os.path.exists(_MODEL):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/face_landmarker/"
        "face_landmarker/float16/1/face_landmarker.task", _MODEL)
_LM = vision.FaceLandmarker.create_from_options(
    vision.FaceLandmarkerOptions(base_options=BaseOptions(model_asset_path=_MODEL),
                                 num_faces=1))

def landmarks(img):
    # 478 face points + 4 corners; None if no face
    rgb = np.ascontiguousarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    res = _LM.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    if not res.face_landmarks:
        return None
    h, w = img.shape[:2]
    pts = np.array([[p.x * w, p.y * h] for p in res.face_landmarks[0]], np.float64)
    corners = np.array([[0, 0], [w - 1, 0], [0, h - 1], [w - 1, h - 1]], np.float64)
    return np.vstack([pts, corners])

def _warp(src, dst, ts, td):
    rs, rd = cv2.boundingRect(np.float32([ts])), cv2.boundingRect(np.float32([td]))
    if min(rs[2], rs[3], rd[2], rd[3]) <= 0:
        return
    ts2 = [(p[0] - rs[0], p[1] - rs[1]) for p in ts]
    td2 = [(p[0] - rd[0], p[1] - rd[1]) for p in td]
    patch = src[rs[1]:rs[1] + rs[3], rs[0]:rs[0] + rs[2]]
    M = cv2.getAffineTransform(np.float32(ts2), np.float32(td2))
    warped = cv2.warpAffine(patch, M, (rd[2], rd[3]), flags=cv2.INTER_LINEAR,
                            borderMode=cv2.BORDER_REFLECT_101)
    mask = np.zeros((rd[3], rd[2], 3), np.float32)
    cv2.fillConvexPoly(mask, np.int32(td2), (1, 1, 1), cv2.LINE_AA)
    roi = dst[rd[1]:rd[1] + rd[3], rd[0]:rd[0] + rd[2]]
    dst[rd[1]:rd[1] + rd[3], rd[0]:rd[0] + rd[2]] = roi * (1 - mask) + warped * mask

def morph(img1, img2):
    # blend two faces into one seamless morph; None on failure
    img1 = cv2.resize(img1, (MORPH_SIZE, MORPH_SIZE))
    img2 = cv2.resize(img2, (MORPH_SIZE, MORPH_SIZE))
    p1, p2 = landmarks(img1), landmarks(img2)
    if p1 is None or p2 is None:
        return None
    pm = (1 - ALPHA) * p1 + ALPHA * p2
    rect = (0, 0, MORPH_SIZE, MORPH_SIZE)
    sub = cv2.Subdiv2D(rect)
    for p in pm:
        sub.insert((float(np.clip(p[0], 0, MORPH_SIZE - 1)),
                    float(np.clip(p[1], 0, MORPH_SIZE - 1))))
    idx = {(round(x, 1), round(y, 1)): i for i, (x, y) in enumerate(pm)}
    tris = []
    for t in sub.getTriangleList():
        vs = [(t[0], t[1]), (t[2], t[3]), (t[4], t[5])]
        if all(0 <= x < MORPH_SIZE and 0 <= y < MORPH_SIZE for x, y in vs):
            tri = [idx.get((round(x, 1), round(y, 1))) for x, y in vs]
            if None not in tri and len(set(tri)) == 3:
                tris.append(tri)
    w1, w2 = np.zeros_like(img1, np.float32), np.zeros_like(img2, np.float32)
    for i, j, k in tris:
        _warp(img1.astype(np.float32), w1, [p1[i], p1[j], p1[k]], [pm[i], pm[j], pm[k]])
        _warp(img2.astype(np.float32), w2, [p2[i], p2[j], p2[k]], [pm[i], pm[j], pm[k]])
    blend = np.clip((1 - ALPHA) * w1 + ALPHA * w2, 0, 255).astype(np.uint8)
    frame = np.clip(w1, 0, 255).astype(np.uint8)              # background from face 1
    hull = cv2.convexHull(pm[:478].astype(np.int32))
    m = np.zeros((MORPH_SIZE, MORPH_SIZE), np.uint8); cv2.fillConvexPoly(m, hull, 255)
    x, y, bw, bh = cv2.boundingRect(hull)
    try:
        return cv2.seamlessClone(blend, frame, m, (x + bw // 2, y + bh // 2),
                                 cv2.NORMAL_CLONE)
    except cv2.error:
        return blend

# example
ex = morph(cv2.imread(face_paths[0]), cv2.imread(face_paths[1]))
fig, ax = plt.subplots(1, 3, figsize=(9, 3.2))
for a, im, t in zip(ax, [face_paths[0], face_paths[1], None],
                    ["identity A", "identity B", "morph (A+B)"]):
    img = ex if im is None else cv2.imread(im)
    a.imshow(cv2.cvtColor(cv2.resize(img, (MORPH_SIZE, MORPH_SIZE)), cv2.COLOR_BGR2RGB))
    a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()


## 5 · Build the dataset and splits

Generate the morphs, pair them with real FFHQ faces, and split into
**train / validation / test** (70 / 15 / 15). Each FFHQ image is a different
person, so no identity leaks between splits.


In [ ]:
# generate morphs from random FFHQ pairs
morph_paths, src = [], iter(range(len(face_paths)))
pbar = tqdm(total=N_MORPH, desc="generating morphs")
i = N_REAL                                   # reals use the first N_REAL faces
while len(morph_paths) < N_MORPH and i + 1 < len(face_paths):
    m = morph(cv2.imread(face_paths[i]), cv2.imread(face_paths[i + 1]))
    i += 2
    if m is None:
        continue
    fp = f"{MORPH_DIR}/morph_{len(morph_paths):05d}.jpg"
    cv2.imwrite(fp, m); morph_paths.append(fp); pbar.update(1)
pbar.close()

real_paths = face_paths[:N_REAL]
items = [(p, 0) for p in real_paths] + [(p, 1) for p in morph_paths]   # 0=real 1=morph
random.shuffle(items)

n = len(items); a, b = int(0.70 * n), int(0.85 * n)
splits = {"train": items[:a], "val": items[a:b], "test": items[b:]}
for s, v in splits.items():
    r = sum(1 for _, y in v if y == 0); m = len(v) - r
    print(f"{s:5s}: {len(v):5d}  (real {r}, morph {m})")

# sample grid
fig, ax = plt.subplots(2, 6, figsize=(15, 5))
for row, (lab, name) in enumerate([(0, "real"), (1, "morph")]):
    samp = [p for p, y in items if y == lab][:6]
    for a_, p in zip(ax[row], samp):
        a_.imshow(cv2.cvtColor(cv2.resize(cv2.imread(p), (MORPH_SIZE, MORPH_SIZE)),
                               cv2.COLOR_BGR2RGB))
        a_.set_title(name); a_.axis("off")
plt.tight_layout(); plt.show()


## 6 · Preprocessing — resize + CLAHE

EfficientNet-B6 needs a fixed **528×528** input. **CLAHE** (Contrast Limited
Adaptive Histogram Equalization) on the lightness channel boosts local contrast
so subtle morph seams become clearer, without amplifying noise. The **same**
steps apply to real and morph images.


In [ ]:
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def preprocess(img):
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    return cv2.cvtColor(cv2.merge((_clahe.apply(l), a, b)), cv2.COLOR_LAB2BGR)

demo = cv2.imread(items[0][0])
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(cv2.cvtColor(cv2.resize(demo, (IMG_SIZE, IMG_SIZE)), cv2.COLOR_BGR2RGB))
ax[0].set_title("resized"); ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(preprocess(demo), cv2.COLOR_BGR2RGB))
ax[1].set_title("resized + CLAHE"); ax[1].axis("off")
plt.tight_layout(); plt.show()


## 7 · Feature extraction — EfficientNet-B6

We load EfficientNet-B6 (ImageNet weights, **no classifier head**) with **Global
Average Pooling**, turning every image into a fixed **2304-dimensional** feature
vector $F$ that captures facial structure, texture, and morphing artefacts.

$$F = \tfrac{1}{N}\sum_i X_L^{(i)} \in \mathbb{R}^{2304}$$


In [ ]:
from tensorflow.keras.applications import EfficientNetB6
from tensorflow.keras.applications.efficientnet import preprocess_input

backbone = EfficientNetB6(include_top=False, weights="imagenet",
                          pooling="avg", input_shape=(IMG_SIZE, IMG_SIZE, 3))
print("feature vector length:", backbone.output_shape[-1])

def extract(pairs, batch=16):
    X, y = [], [p[1] for p in pairs]
    paths = [p[0] for p in pairs]
    for i in tqdm(range(0, len(paths), batch), desc="extracting"):
        imgs = [preprocess_input(cv2.cvtColor(preprocess(cv2.imread(p)),
                                              cv2.COLOR_BGR2RGB).astype("float32"))
                for p in paths[i:i + batch]]
        X.append(backbone.predict(np.stack(imgs), verbose=0))
    return np.concatenate(X), np.array(y)

feats = {s: extract(v) for s, v in splits.items()}
for s, (X, y) in feats.items():
    print(f"{s:5s}: features {X.shape}")


## 8 · Classifier — SVM

We standardize the features and train a **Support Vector Machine** (RBF kernel),
which finds the maximum-margin boundary between real and morph:

$$\min_{w,b}\ \tfrac12\lVert w\rVert^2 + C\sum_i \xi_i \quad\text{s.t.}\quad y_i(w^{\!\top}F_i+b)\ge 1-\xi_i$$


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

scaler = StandardScaler().fit(feats["train"][0])
Xtr = scaler.transform(feats["train"][0]); ytr = feats["train"][1]
Xva = scaler.transform(feats["val"][0]);   yva = feats["val"][1]
Xte = scaler.transform(feats["test"][0]);  yte = feats["test"][1]

svm = SVC(kernel="rbf", C=10.0, gamma="scale", probability=True, random_state=SEED)
svm.fit(Xtr, ytr)
print("train accuracy:", round(accuracy_score(ytr, svm.predict(Xtr)), 3))
print("val   accuracy:", round(accuracy_score(yva, svm.predict(Xva)), 3))


## 9 · Results

Evaluate on the held-out **test** set: confusion matrix, the standard metrics,
and the ROC curve with its AUC.


In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, auc, accuracy_score,
                             precision_score, recall_score, f1_score)

pred  = svm.predict(Xte)
proba = svm.predict_proba(Xte)[:, 1]
cm = confusion_matrix(yte, pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
im = ax[0].imshow(cm, cmap="Blues"); plt.colorbar(im, ax=ax[0])
ax[0].set(title="Confusion Matrix", xticks=[0, 1], yticks=[0, 1],
          xlabel="Predicted", ylabel="Actual")
ax[0].set_xticklabels(["Real", "Morph"]); ax[0].set_yticklabels(["Real", "Morph"])
for i in range(2):
    for j in range(2):
        ax[0].text(j, i, cm[i, j], ha="center", va="center", fontsize=15,
                   color="white" if cm[i, j] > cm.max() / 2 else "black")

fpr, tpr, _ = roc_curve(yte, proba); roc_auc = auc(fpr, tpr)
ax[1].plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.3f}")
ax[1].plot([0, 1], [0, 1], "--", color="gray")
ax[1].set(title="ROC Curve", xlabel="False Positive Rate", ylabel="True Positive Rate")
ax[1].legend(loc="lower right"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Accuracy : {accuracy_score(yte, pred):.3f}")
print(f"Precision: {precision_score(yte, pred):.3f}")
print(f"Recall   : {recall_score(yte, pred):.3f}")
print(f"F1-score : {f1_score(yte, pred):.3f}")
print(f"AUC      : {roc_auc:.3f}")
print(f"\nTP={tp}  TN={tn}  FP={fp}  FN={fn}")
print("\n" + classification_report(yte, pred, target_names=["Real", "Morph"]))
print("Paper reference: 89.9% accuracy, AUC 0.965.")


## 10 · Try it on your own image

Upload a face photo and the model predicts **Real** or **Morph** with a
confidence score.


In [ ]:
from google.colab import files

def predict(img):
    f = backbone.predict(preprocess_input(
        cv2.cvtColor(preprocess(img), cv2.COLOR_BGR2RGB).astype("float32")[None]),
        verbose=0)
    p = float(svm.predict_proba(scaler.transform(f))[0, 1])
    return ("MORPH" if p >= 0.5 else "REAL"), p

for name, data in files.upload().items():
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    label, p = predict(img)
    plt.figure(figsize=(4, 4))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"{label}   (morph prob = {p:.2f})",
              color="red" if label == "MORPH" else "green", fontweight="bold")
    plt.axis("off"); plt.show()


## Summary

| Stage | Method |
|---|---|
| Real faces | FFHQ |
| Morphs | Landmark morphing (Delaunay + affine + seamless clone) |
| Preprocessing | Resize 528 + CLAHE |
| Features | EfficientNet-B6 → 2304-d (Global Average Pooling) |
| Classifier | SVM (RBF), maximum margin |
| Metrics | Accuracy, Precision, Recall, F1, ROC-AUC |

The two-stage design — deep features + SVM — keeps the model accurate and
resistant to overfitting on a modest dataset, matching the DMorphNet paper
(89.9% accuracy, AUC 0.965). Set `DEMO = False` for the full-scale run.
